In [234]:
# Setup
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Define path to data folder
DATA_DIR = Path.cwd().parents[2] / 'data' / 'spreadsheets'

# Read in CD-POS Population Data
pop2020 = pd.read_excel(DATA_DIR / 'nga_admpop_2020.xlsx', sheet_name='nga_admpop_adm2_2020')

# Read in CityPopulation.de 2022 Population Projection Data
city_pop22 = pd.read_excel(DATA_DIR / 'NGA_citypop_pop2022.xlsx', sheet_name='Sheet1')

# Read in COD-PS Nigeria Intrinsic Population Growth Rate Data
cod = pd.read_excel(DATA_DIR / 'NGA_PGR_2022.xlsx', sheet_name='PGR_Proj')
cod = cod.rename(columns={
    'PGR_BOTH_2022': 'ANN_POP_CHANGE'
})

# Read in City Population Annual Population Change (2006-2022) data
city = pd.read_excel(DATA_DIR / 'NGA_citypop_apc.xlsx', sheet_name='cp_apc')

# Read in raw LGA-SENATORIAL DISTRICT-FEDERAL CONSTITUENCY CROSSWALK
pol_map = pd.read_excel(DATA_DIR / 'nga_lga_pol_map.xlsx')

# Merge COD-PS Intrinsic Population Growth Rate Data with City Population Annual Population Change Rate Data for comparison
pop_growth_merged = city.merge(cod, on='ADM1_NAME', how='outer', suffixes=('_city', '_cod'))
print(pop_growth_merged.head())


   ADM1_NAME  ANN_POP_CHANGE_city  ANN_POP_CHANGE_cod
0       Abia                  2.4                2.35
1    Adamawa                  2.7                2.71
2  Akwa Ibom                  1.5                1.52
3    Anambra                  2.2                2.21
4     Bauchi                  3.7                3.62


In [235]:
# Identify missing states in COD-PS data vs City Population data
missing_in_cod = pop_growth_merged[pop_growth_merged['ANN_POP_CHANGE_cod'].isna()][['ADM1_NAME', 'ANN_POP_CHANGE_city']]
print("States missing from COD-PS:\n", missing_in_cod)

missing_in_city = pop_growth_merged[pop_growth_merged['ANN_POP_CHANGE_city'].isna()][['ADM1_NAME', 'ANN_POP_CHANGE_cod']]
print("\nStates missing from City Population data:\n", missing_in_city)

States missing from COD-PS:
    ADM1_NAME  ANN_POP_CHANGE_city
25  Nasarawa                  2.8

States missing from City Population data:
 Empty DataFrame
Columns: [ADM1_NAME, ANN_POP_CHANGE_cod]
Index: []


Nasarawa is missing data in COD-PS. We will measure sameness/difference in the data next to determine if we can fill in missing COD-PS data with City Population data.

In [236]:
# Measure sameness/difference in rates data

# Work with states that have data points in both tables
both = pop_growth_merged.dropna(subset=['ANN_POP_CHANGE_city', 'ANN_POP_CHANGE_cod']).copy()

# Absolute Difference
both['abs_diff'] = (both['ANN_POP_CHANGE_city'] - both['ANN_POP_CHANGE_cod']).abs()

# Check if they match within a rounding tolerance
both['within_rounding_tolerance'] = both['abs_diff'] <= 0.1

# Summary metrics
print("Max absolute difference:   ", both['abs_diff'].max())
print("Mean absolute difference:  ", both['abs_diff'].mean())
print("States within rounding:    ", both['within_rounding_tolerance'].sum(), "of", len(both))
print("Correlation:               ", both['ANN_POP_CHANGE_city'].corr(both['ANN_POP_CHANGE_cod']).round(4))

# See any states that diverge beyond rounding
outliers = both[~both['within_rounding_tolerance']]
print("\nStates outside rounding tolerance:\n", outliers[['ADM1_NAME', 'ANN_POP_CHANGE_city', 'ANN_POP_CHANGE_cod', 'abs_diff']])

Max absolute difference:    5.2
Mean absolute difference:   0.19194444444444442
States within rounding:     33 of 36
Correlation:                0.7154

States outside rounding tolerance:
                     ADM1_NAME  ANN_POP_CHANGE_city  ANN_POP_CHANGE_cod  \
14  Federal Capital Territory                  5.0                4.88   
17                     Jigawa                  3.5                3.40   
36                    Zamfara                  3.7                8.90   

    abs_diff  
14      0.12  
17      0.10  
36      5.20  


The annual population change data between the two datasets appears to be the same. We will fill in the missing Nasarawa COD-PS rate with the City Population rate.

In our outlier check, FCT (Abuja), Jigawa, and Zamfara are flagged as outliers. While FCT and Jigawa look like rounding errors (5.0 vs 4.88 and 3.5 vs 3.4 respectively in City Population vs COD-PS data), Zamfara looks like completely different rates (3.7 vs 8.9 in City Population vs COD-PS data). We will trust the City Population data. This will potentially underestimate our population growth.

In [237]:
# Decision: fill missing COD-PS rate for Nasarawa with City Population rates
#pop_growth_merged['ANN_POP_CHANGE_cod'] = pop_growth_merged['ANN_POP_CHANGE_cod'].fillna(pop_growth_merged['ANN_POP_CHANGE_city'])
pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Nasarawa', 'ANN_POP_CHANGE_cod'] = pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Nasarawa', 'ANN_POP_CHANGE_city']

# Flag Zamfara discrepancy
zamfara = pop_growth_merged[pop_growth_merged['ADM1_NAME'] == 'Zamfara']
print(zamfara[['ADM1_NAME', 'ANN_POP_CHANGE_city', 'ANN_POP_CHANGE_cod']])

# Option A: Trust City Population
pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Zamfara', 'rate_final'] = 3.7

# Option B: Trust COD-PS
#pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Zamfara', 'rate_final'] = 8.9

# Final Reconciled Column
pop_growth_merged['rate_final'] = pop_growth_merged['rate_final'].fillna(pop_growth_merged['ANN_POP_CHANGE_cod'])

# Sanity check - no nulls
print("\nFinal population growth rates:\n", pop_growth_merged[['ADM1_NAME', 'rate_final']])

   ADM1_NAME  ANN_POP_CHANGE_city  ANN_POP_CHANGE_cod
36   Zamfara                  3.7                 8.9

Final population growth rates:
                     ADM1_NAME  rate_final
0                        Abia        2.35
1                     Adamawa        2.71
2                   Akwa Ibom        1.52
3                     Anambra        2.21
4                      Bauchi        3.62
5                     Bayelsa        2.49
6                       Benue        2.30
7                       Borno        2.39
8                 Cross River        2.63
9                       Delta        1.97
10                     Ebonyi        2.49
11                        Edo        2.44
12                      Ekiti        2.53
13                      Enugu        2.26
14  Federal Capital Territory        4.88
15                      Gombe        3.22
16                        Imo        2.06
17                     Jigawa        3.40
18                     Kaduna        2.44
19                 

In [238]:
# Create a copy of pop2020 with only ADM1_NAME, ADM2_NAME, and T_TL, and rename T_TL to POP_2020
#pop2020_clean = pop2020[['ADM1_NAME', 'ADM2_NAME', 'T_TL']].rename(columns={'T_TL': 'POP_2020'}).copy()
#print(pop2020_clean.head())
# Count number of unique states/lgas in pop22 dataset
#print("Unique states in pop2020_clean:", pop2020_clean['ADM1_NAME'].nunique())
#print("Unique LGAs in pop2020_clean:   ", pop2020_clean['ADM2_NAME'].nunique())

# Create a copy of pop2022 with only ADM1_NAME, LGA, and POPULATION
pop2022_clean = city_pop22.rename(columns={
    'STATE': 'ADM1_NAME',
    'POP_2022': 'POP22'
}).copy()

# Count number of unique states/lgas in pop22 dataset
print("Unique states in pop2022_clean:", pop2022_clean['ADM1_NAME'].nunique())
print("Unique LGAs in pop2022_clean:   ", pop2022_clean['LGA'].nunique())

# Create a table to count number of rows by ADM1_NAME in pop2022_clean to identify which states have missing LGAs
lga_counts = pop2022_clean.groupby('ADM1_NAME')['LGA'].count().reset_index().rename(columns={'LGA': 'LGA_COUNT'})
print(lga_counts)
# Note, only 769 instead of 774 LGAs in the pop2022 dataset. Checked count of LGAs by state and the counts are as expected. Issue is probably because of LGAs in different states with the same name

# Standardize ADM1_NAME in pop_growth_merged and pop2020_clean to uppercase for comparison
states_growth = set(pop_growth_merged['ADM1_NAME'].str.upper())
states_pop    = set(pop2022_clean['ADM1_NAME'].str.upper())

# Check for any mismatches
print("In growth but not in pop2022:", states_growth - states_pop)
print("In pop2022 but not in growth:", states_pop - states_growth)

# It looks like the state names match in both datasets
pop_growth_merged['ADM1_NAME'] = pop_growth_merged['ADM1_NAME'].str.upper()

# Merge
merged = pop2022_clean.merge(pop_growth_merged, on='ADM1_NAME', how='left')
print(merged.head())

Unique states in pop2022_clean: 37
Unique LGAs in pop2022_clean:    769
                    ADM1_NAME  LGA_COUNT
0                        ABIA         17
1                     ADAMAWA         21
2                   AKWA IBOM         31
3                     ANAMBRA         21
4                      BAUCHI         20
5                     BAYELSA          8
6                       BENUE         23
7                       BORNO         27
8                 CROSS RIVER         18
9                       DELTA         25
10                     EBONYI         13
11                        EDO         18
12                      EKITI         16
13                      ENUGU         17
14  FEDERAL CAPITAL TERRITORY          6
15                      GOMBE         11
16                        IMO         27
17                     JIGAWA         27
18                     KADUNA         23
19                       KANO         44
20                    KATSINA         34
21                      KE

In [239]:
merged_final = merged[['ADM1_NAME', 'LGA', 'POP22', 'rate_final']].rename(columns={'rate_final': 'ANNUAL_GROWTH_RATE22'}).copy()
merged_final['POP25_PROJECTED'] = merged_final['POP22'] * (1 + merged_final['ANNUAL_GROWTH_RATE22']/100)**3

#merged.to_csv(DATA_DIR / 'nga_lga_cleaned.csv', index=False)
merged_final.to_excel(DATA_DIR / 'nga_pop_estimates.xlsx', index=False, sheet_name='popest_lga')

In [240]:
# Clean Political Map DataFrame

# Step 1: Extract State from Senatorial District
directions = ['North', 'South', 'East', 'West', 'Central', 'North East', 'North West', 'South East', 'South West']
pattern = r'\s+(' + '|'.join(directions) + r')$'

pol_map['State'] = pol_map['Senatorial District'].str.replace(pattern, '', regex=True).str.strip()
pol_map['State'] = pol_map['State'].replace('FCT', 'Federal Capital Territory')

# Step 2: Split Constituency (LGAs) by "/" and explode into separate rows
pol_map['LGA'] = pol_map['Constituency'].str.split('/')

pol_map_long = pol_map.explode('LGA')

# Step 3: Clean up and rename columns
pol_map_long['LGA'] = pol_map_long['LGA'].str.strip()

pol_map_final = pol_map_long.rename(columns={
    'State':               'STATE',
    'Senatorial District': 'SEN_DIST',
    'Constituency':        'FED_CONST'
})[['STATE', 'LGA', 'SEN_DIST', 'FED_CONST']]

# Step 4: Drop duplicates and reset index
pol_map_final = pol_map_final.drop_duplicates().reset_index(drop=True)

print(pol_map_final.shape)
print(pol_map_final.head(10))
#print(pol_map_final)

# Sanity check: count number of unique states/lgas in pol_map_final dataset
print("Unique LGAs:  ", pol_map_final['LGA'].nunique())
print("Unique States:", pol_map_final['STATE'].nunique())

print(pol_map_final['STATE'].unique())

(793, 4)
  STATE                LGA      SEN_DIST                            FED_CONST
0  Abia  Isiala Ngwa North  Abia Central  Isiala Ngwa North/Isiala Ngwa South
1  Abia  Isiala Ngwa South  Abia Central  Isiala Ngwa North/Isiala Ngwa South
2  Abia            Obingwa  Abia Central           Obingwa/Ugwunagbo/Osisioma
3  Abia          Ugwunagbo  Abia Central           Obingwa/Ugwunagbo/Osisioma
4  Abia           Osisioma  Abia Central           Obingwa/Ugwunagbo/Osisioma
5  Abia      Umuahia North  Abia Central  Umuahia North/Umuahia South/Ikwuano
6  Abia      Umuahia South  Abia Central  Umuahia North/Umuahia South/Ikwuano
7  Abia            Ikwuano  Abia Central  Umuahia North/Umuahia South/Ikwuano
8  Abia          Arochukwu    Abia North                     Arochukwu/Ohafia
9  Abia             Ohafia    Abia North                     Arochukwu/Ohafia
Unique LGAs:   779
Unique States: 37
['Abia' 'Adamawa' 'Akwa Ibom' 'Anambra' 'Bauchi' 'Benue' 'Borno'
 'Cross River' 'Ebonyi' 'Ekiti'

In [266]:
# Some LGAs such as Isolo, Lagos Island, Mushin, Surulere in Lagos and Port Harcourt in Rivers are split. So there's Isolo I/II, Mushin I/II, Port Harcourt I/II, etc. Without sub-LGA population data, we won't be able to split these LGAs correctly. We will assume a 50/50 split of the population for these LGAs.
pol_map_final['IS_SPLIT'] = pol_map_final['LGA'].str.contains(r'\s+I{1,2}$', regex=True)

print(pol_map_final[pol_map_final['IS_SPLIT']])

# Create a clean LGA name that strips I/II for merging with the LGA-level population data
pol_map_final['LGA_MERGE'] = pol_map_final['LGA'].str.replace(r'\s+I{1,2}$', '', regex=True).str.strip()

# Use LGA_MERGE to join population data


# Use LGA_MERGE to join population data
pol_merged = pol_map_final.merge(merged_final[['ADM1_NAME', 'LGA', 'POP25_PROJECTED']],
                             left_on=['STATE', 'LGA_MERGE'],
                             right_on=['ADM1_NAME', 'LGA'],
                             how='left')

# For split LGAs, divide the population by 2; for non-split, keep as is
pol_merged['POP25_CONST'] = pol_merged.apply(
    lambda row: row['POP25_PROJECTED'] / 2 if row['IS_SPLIT'] else row['POP25_PROJECTED'],
    axis=1
)

# Aggregate to federal constituency level
const_pop = pol_merged.groupby(['STATE', 'FED_CONST'], as_index=False)['POP25_CONST'].sum()


      STATE               LGA       SEN_DIST         FED_CONST  IS_SPLIT  \
456   Lagos    Lagos Island I  Lagos Central    Lagos Island I      True   
457   Lagos   Lagos Island II  Lagos Central   Lagos Island II      True   
459   Lagos        Surulere I  Lagos Central        Surulere I      True   
460   Lagos       Surulere II  Lagos Central       Surulere II      True   
474   Lagos          Mushin I     Lagos West          Mushin I      True   
475   Lagos         Mushin II     Lagos West         Mushin II      True   
478   Lagos           Isolo I     Lagos West    Oshodi/Isolo I      True   
480   Lagos          Isolo II     Lagos West   Oshodi/Isolo II      True   
649  Rivers   Port Harcourt I    Rivers East   Port Harcourt I      True   
650  Rivers  Port Harcourt II    Rivers East  Port Harcourt II      True   

         LGA_CLEAN  
456   Lagos Island  
457   Lagos Island  
459       Surulere  
460       Surulere  
474         Mushin  
475         Mushin  
478          Iso

In [272]:
print(const_pop.head(20))

        STATE                            FED_CONST  POP25_CONST
0        Abia                  Aba North/Aba South          0.0
1        Abia                     Arochukwu/Ohafia          0.0
2        Abia                                Bende          0.0
3        Abia  Isiala Ngwa North/Isiala Ngwa South          0.0
4        Abia                Isuikwuato/Umunneochi          0.0
5        Abia           Obingwa/Ugwunagbo/Osisioma          0.0
6        Abia                  Ukwa East/Ukwa West          0.0
7        Abia  Umuahia North/Umuahia South/Ikwuano          0.0
8     Adamawa                  Demsa/Numan/Lamurde          0.0
9     Adamawa                          Fufore/Song          0.0
10    Adamawa                       Guyuk/Shelleng          0.0
11    Adamawa                           Hong/Gombi          0.0
12    Adamawa         Jada/Ganye/Mayo Belwa/Toungo          0.0
13    Adamawa                     Michika/Madagali          0.0
14    Adamawa          Mubi North/Mubi S

In [ ]:
# Add to new sheet in nga_pop_estimates.xlsx
const_pop.to_excel(DATA_DIR / 'nga_pop_estimates.xlsx', index=False, sheet_name='popest_const')